# Phase 4 Feature Exploration
## HFTExperiment v2 | 2026-05-11

Candidates: GKV (P6), d_ATR (P1), 5-bar momentum (P7), DHPF partitions (P7)
Evaluation: KS + MI + Redundancy

## 0. Setup + Load NPZ

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

# --- Path resolution: Windows local and Colab ---
def find_npz():
    candidates = [
        Path("data/training_ready.npz"),
        Path("../data/training_ready.npz"),
        Path("training_ready.npz"),
        Path("D:/HFTExperiment/data/training_ready.npz"),
        Path.home() / "HFTExperiment/data/training_ready.npz",
        Path("/content/drive/MyDrive/Colab Notebooks/training_ready.npz"),
    ]
    for p in candidates:
        if p.exists():
            print(f"Found: {p}"); return str(p)
    raise FileNotFoundError("training_ready.npz not found. Set NPZ_PATH manually.")

NPZ_PATH = find_npz()
_d = np.load(NPZ_PATH, allow_pickle=True)
print("Keys:", list(_d.keys()))

labels       = _d["labels"].astype(int)
raw_features = _d["features"]

# Handle 2D (N,10) or 3D (N,seq,10)
if raw_features.ndim == 3:
    seq_len = raw_features.shape[1]
    n_seq   = raw_features.shape[0]
    if len(labels) > n_seq:
        labels = labels[seq_len-1 : seq_len-1+n_seq]
    fb    = raw_features[:, -1, :]
    feats = raw_features
    print(f"3D: {raw_features.shape} -> last-bar fb: {fb.shape}")
elif raw_features.ndim == 2:
    fb    = raw_features
    feats = raw_features[:, np.newaxis, :]
    n     = min(len(labels), len(fb))
    labels = labels[:n]; fb = fb[:n]; feats = feats[:n]
    print(f"2D: {raw_features.shape} -> using as fb directly")
else:
    raise ValueError(f"Unexpected features shape: {raw_features.shape}")

FEAT_NAMES = ["open_sc","high_sc","low_sc","close_sc","vol_sc",
              "spread_sc","bar_return_bps","wick_asymmetry","vol_zscore","spread_pressure"]

assert len(fb)==len(labels), f"Length mismatch: fb={len(fb)} labels={len(labels)}"
sell_mask = labels==0; hold_mask = labels==1; buy_mask = labels==2
print(f"Sell={sell_mask.sum():,} ({100*sell_mask.mean():.2f}%) "
      f"Hold={hold_mask.sum():,}  Buy={buy_mask.sum():,}")


Found: ..\data\training_ready.npz
Keys: ['features', 'labels', 'close', 'high', 'low', 'timestamps_ns', 'gmm2', 'km_enc', 'vol_enc', 'gs_q', 'cu_au', 'rq', 'atr_norm', 'trend_norm', 'session_phase', 'tick_volume_raw', 'metadata']
2D: (5680771, 10) -> using as fb directly
Sell=199,523 (3.51%) Hold=5,439,333  Buy=41,915


## Helper: eval_feature()

In [2]:
# eval_feature: KS + MI + Redundancy
# values: 1D float array aligned with labels
# fb: (N, n_feat) last-bar feature matrix
def eval_feature(name, values, labels, fb, thresh_mi=0.005, thresh_red=10.0):
    values = np.asarray(values, dtype=np.float64).ravel()
    assert len(values)==len(labels)==len(fb)

    ks_sell, _ = ks_2samp(values[labels==0], values[labels==1])
    ks_buy,  _ = ks_2samp(values[labels==2], values[labels==1])
    ks_max = max(ks_sell, ks_buy)

    vol_z = fb[:, min(8, fb.shape[1]-1)].astype(float)
    hv = vol_z > np.percentile(vol_z, 80)
    ks_regime = ks_2samp(values[hv], values[~hv])[0] if (hv.sum()>100 and (~hv).sum()>100) else 0.0

    # MI vs sell label -- X must be (N,1) float, y must be 1D int
    bin_labels = (labels==0).astype(np.int32)
    mi = mutual_info_classif(
        values.reshape(-1,1), bin_labels,
        discrete_features=False, random_state=42
    )[0]

    n_base = min(6, fb.shape[1])
    mi_base = np.mean([
        mutual_info_classif(fb[:,j:j+1].astype(float), bin_labels,
                           discrete_features=False, random_state=42)[0]
        for j in range(n_base)
    ]) + 1e-9

    # Redundancy: MI between candidate and each existing feature
    max_pair = 1e-9; best_feat = "none"
    for j in range(fb.shape[1]):
        mi_j = mutual_info_regression(
            fb[:,j:j+1].astype(float), values,
            discrete_features=False, random_state=42
        )[0]
        if mi_j > max_pair:
            max_pair = mi_j
            best_feat = FEAT_NAMES[j] if j<len(FEAT_NAMES) else f"feat[{j}]"

    ratio  = max_pair / (mi + 1e-9)
    adj_mi = max(0.0, mi - max_pair)

    verdict = "EXCLUDE"
    if ks_max>0.05 and mi>thresh_mi and ratio<thresh_red: verdict = "SUPERVISED"
    elif ks_max>0.05 and mi>thresh_mi: verdict = "RL OBS"
    elif ks_max>0.05: verdict = "EXCLUDE (MI fails)"

    sig = "STRONG" if ks_max>0.10 else ("MOD" if ks_max>0.05 else "WEAK")
    print(f"\n{'='*55}\nFeature: {name}")
    print(f"  KS sell={ks_sell:.4f} buy={ks_buy:.4f} [{sig}] | regime={ks_regime:.4f}")
    print(f"  MI={mi:.6f} ({mi/mi_base:.2f}x OHLCV) {'PASS' if mi>thresh_mi else 'FAIL'}")
    print(f"  MaxPairMI={max_pair:.5f} vs {best_feat} | ratio={ratio:.2f} {'PASS' if ratio<thresh_red else 'FAIL'}")
    print(f"  adj_MI={adj_mi:.6f} | VERDICT: {verdict}")
    return {"name":name,"ks_sell":ks_sell,"ks_buy":ks_buy,"ks_regime":ks_regime,
            "mi":mi,"mi_vs_ohlcv":mi/mi_base,"ratio":ratio,"adj_mi":adj_mi,
            "best_feat":best_feat,"verdict":verdict}

results = {}
print("Helper ready.")


Helper ready.


## 1. Garman-Klass Volatility (GKV)
P6 Goel et al 2025: `GKV = 0.5*(ln H/L)^2 - (2ln2-1)*(ln C/O)^2`
Candidate to replace or augment vol_zscore (feature[8])

In [3]:
# Garman-Klass Volatility: GKV = 0.5*(ln H/L)^2 - (2ln2-1)*(ln C/O)^2
# P6 Goel et al 2025: richer intraday vol signal than close-to-close
o = fb[:,0].astype(float); h = fb[:,1].astype(float)
l = fb[:,2].astype(float); c = fb[:,3].astype(float)
eps = 1e-4; off = 2.0
gkv_raw  = (0.5*np.log((h+off)/(l+off+eps))**2
            - (2*np.log(2)-1)*np.log((c+off)/(o+off+eps))**2)
gkv_norm = np.tanh((gkv_raw - np.median(gkv_raw))/(np.std(gkv_raw)+1e-8))
print(f"GKV: mean={gkv_norm.mean():.4f} std={gkv_norm.std():.4f}")

vol_z = fb[:,8].astype(float)
fig, axes = plt.subplots(1,3,figsize=(15,4))
for mask,name_,c_ in [(sell_mask,'sell','red'),(hold_mask,'hold','blue'),(buy_mask,'buy','green')]:
    axes[0].hist(gkv_norm[mask],bins=60,alpha=0.5,label=name_,color=c_)
axes[0].set_title("GKV by label"); axes[0].legend()
for mask,name_,c_ in [(sell_mask,'sell','red'),(hold_mask,'hold','blue'),(buy_mask,'buy','green')]:
    axes[1].hist(vol_z[mask],bins=60,alpha=0.5,label=name_,color=c_)
axes[1].set_title("vol_zscore (existing)"); axes[1].legend()
n_s = min(3000,len(labels))
idx = np.random.choice(len(labels),n_s,replace=False)
cols_ = ['red' if labels[i]==0 else 'blue' if labels[i]==1 else 'green' for i in idx]
axes[2].scatter(vol_z[idx],gkv_norm[idx],c=cols_,alpha=0.3,s=4)
axes[2].set_xlabel("vol_zscore"); axes[2].set_ylabel("GKV")
axes[2].set_title("GKV vs vol_zscore by label")
plt.tight_layout(); plt.savefig("gkv_analysis.png",dpi=100); plt.show()
print("Saved: gkv_analysis.png")
results["gkv_norm"] = eval_feature("gkv_norm", gkv_norm, labels, fb)


GKV: mean=0.0844 std=0.3890
Saved: gkv_analysis.png

Feature: gkv_norm
  KS sell=0.0730 buy=0.0807 [MOD] | regime=0.1240
  MI=0.000598 (0.06x OHLCV) FAIL
  MaxPairMI=0.21582 vs bar_return_bps | ratio=360.94 FAIL
  adj_MI=0.000000 | VERDICT: EXCLUDE (MI fails)


## 2. Directional ATR (d_ATR)
P1 Ma et al 2021: rising ATR more predictive than flat-high ATR.
Deploy gate improvement: only block Bear+HIGH when ATR is actively rising.

In [4]:
# Directional ATR -- P1 Ma et al 2021
# d_ATR > 0 means ATR actively rising (higher uncertainty ahead)
# Deploy gate improvement: only block Bear+HIGH when ATR is rising
if feats.ndim==3 and feats.shape[1]>=20:
    atrs   = feats[:,-20:,1].astype(float) - feats[:,-20:,2].astype(float)
    atr_rec  = atrs[:,-5:].mean(axis=1)
    atr_base = atrs[:,-15:-5].mean(axis=1)
else:
    print("2D NPZ: using |wick_asymmetry| as ATR proxy")
    atr_rec  = np.abs(fb[:,7]).astype(float)
    atr_base = np.full(len(fb), np.abs(fb[:,7]).mean())

d_atr = np.tanh((atr_rec - atr_base)/(np.abs(atr_base)+1e-8)*5)
print(f"d_ATR: mean={d_atr.mean():.4f} std={d_atr.std():.4f}")

ret   = fb[:,6].astype(float); vol_z = fb[:,8].astype(float)
bear  = ret < 0; high = vol_z > np.percentile(vol_z,75); rising = d_atr > 0.1
bear_high = bear & high
bhr = (bear_high & rising).sum(); bhf = (bear_high & ~rising).sum()

fig, axes = plt.subplots(1,2,figsize=(12,4))
for mask,name_,c_ in [(sell_mask,'sell','red'),(hold_mask,'hold','blue'),(buy_mask,'buy','green')]:
    axes[0].hist(d_atr[mask],bins=60,alpha=0.5,label=name_,color=c_)
axes[0].set_title("d_ATR direction by label"); axes[0].legend()
axes[1].bar(['Rising ATR\n(block)','Falling ATR\n(allow)'],[bhr,bhf],color=['red','green'])
for i,v in enumerate([bhr,bhf]):
    axes[1].text(i,v*0.5,f"{v:,}\n({100*v/max(bear_high.sum(),1):.0f}%)",
                ha='center',va='center',fontweight='bold',color='white')
axes[1].set_title(f"Bear+HIGH gate refinement (P1)\nTotal={bear_high.sum():,}")
plt.tight_layout(); plt.savefig("d_atr_analysis.png",dpi=100); plt.show()
print(f"Gate: block {bhr:,} rising / allow {bhf:,} falling of {bear_high.sum():,} Bear+HIGH bars")
results["d_atr_norm"] = eval_feature("d_atr_norm", d_atr, labels, fb)


2D NPZ: using |wick_asymmetry| as ATR proxy
d_ATR: mean=0.0698 std=0.8901
Gate: block 9,758 rising / allow 11,321 falling of 21,079 Bear+HIGH bars

Feature: d_atr_norm
  KS sell=0.1265 buy=0.1489 [STRONG] | regime=0.1775
  MI=0.049595 (5.34x OHLCV) PASS
  MaxPairMI=6.80226 vs wick_asymmetry | ratio=137.16 FAIL
  adj_MI=0.000000 | VERDICT: RL OBS


## 3. 5-bar Momentum
P7 Zhao et al 2025: lag features improve gold price prediction.

In [5]:
# 5-bar Momentum -- P7 Zhao et al 2025
if feats.ndim==3 and feats.shape[1]>=6:
    c_now  = feats[:,-1,3].astype(float)
    c_lag5 = feats[:,-6,3].astype(float)
    mom5   = np.tanh((c_now - c_lag5)/(np.abs(c_lag5)+1e-8)*20)
else:
    print("2D NPZ: using bar_return_bps as momentum proxy (degenerate)")
    mom5 = fb[:,6].astype(float)

print(f"mom5: mean={mom5.mean():.4f} std={mom5.std():.4f}")
fig, ax = plt.subplots(figsize=(8,4))
for mask,name_,c_ in [(sell_mask,'sell','red'),(hold_mask,'hold','blue'),(buy_mask,'buy','green')]:
    ax.hist(mom5[mask],bins=60,alpha=0.5,label=name_,color=c_)
ax.set_title("5-bar momentum by label"); ax.legend()
plt.tight_layout(); plt.savefig("mom5_analysis.png",dpi=100); plt.show()
results["momentum_5bar"] = eval_feature("momentum_5bar", mom5, labels, fb)


2D NPZ: using bar_return_bps as momentum proxy (degenerate)
mom5: mean=0.0034 std=3.0643

Feature: momentum_5bar
  KS sell=0.1553 buy=0.1830 [STRONG] | regime=0.1952
  MI=0.012213 (1.31x OHLCV) PASS
  MaxPairMI=13.72706 vs bar_return_bps | ratio=1123.99 FAIL
  adj_MI=0.000000 | VERDICT: RL OBS


## 4. DHPF 4-Partition Validation
P7: Add Bear-SHOCK partition (vol >= 99th pct) with x4.0 multiplier.

In [6]:
# DHPF 4-partition -- P7 Zhao et al 2025
# Does Bear-SHOCK (vol >= 99th pct) have disproportionate sell labels?
vol_z = fb[:,8].astype(float); ret = fb[:,6].astype(float)
bull  = ret > 0
vp25  = np.percentile(vol_z,25); vp75 = np.percentile(vol_z,75); vp99 = np.percentile(vol_z,99)

parts = {
    "Bull-LOW":    bull  & (vol_z<vp25),
    "Bull-NORMAL": bull  & (vol_z>=vp25) & (vol_z<vp75),
    "Bull-HIGH":   bull  & (vol_z>=vp75) & (vol_z<vp99),
    "Bull-SHOCK":  bull  & (vol_z>=vp99),
    "Bear-LOW":   ~bull  & (vol_z<vp25),
    "Bear-NORMAL":~bull  & (vol_z>=vp25) & (vol_z<vp75),
    "Bear-HIGH":  ~bull  & (vol_z>=vp75) & (vol_z<vp99),
    "Bear-SHOCK": ~bull  & (vol_z>=vp99),
}
base_sell = (labels==0).mean()
print(f"Baseline sell: {100*base_sell:.3f}%\n")
print(f"{'Partition':<15} {'N':>8} {'Sell%':>8} {'Hold%':>8} {'Rec.mult':>10}")
print("-"*50)
for pname, pmask in parts.items():
    n = pmask.sum()
    if n==0: continue
    sr = (labels[pmask]==0).mean(); hr = (labels[pmask]==1).mean()
    mult = float(np.clip(base_sell*2/(sr+1e-8),1.0,8.0))
    print(f"  {pname:<13} {n:>8,} {100*sr:>7.3f}% {100*hr:>7.3f}% {mult:>10.2f}x")

pnames = list(parts.keys())
srates = [100*(labels[m]==0).mean() if m.sum()>0 else 0. for m in parts.values()]
fig,ax = plt.subplots(figsize=(12,4))
ax.bar(pnames,srates,color=['green' if 'Bull' in p else 'red' for p in pnames],alpha=0.8)
ax.axhline(y=100*base_sell,color='k',linestyle='--',label=f"Baseline={100*base_sell:.2f}%")
ax.set_ylabel("Sell label rate (%)"); ax.set_title("P7 DHPF: Sell label by partition")
ax.legend(); plt.xticks(rotation=30,ha='right')
plt.tight_layout(); plt.savefig("dhpf_partitions.png",dpi=100); plt.show()
shock = vol_z >= vp99
print(f"\nShock bars (>= 99th): {shock.sum():,} ({100*shock.mean():.2f}%)")
print(f"Sell in shock: {(labels[shock]==0).sum():,} ({100*(labels[shock]==0).mean():.2f}%)")


Baseline sell: 3.512%

Partition              N    Sell%    Hold%   Rec.mult
--------------------------------------------------
  Bull-LOW        26,715  43.657%  38.289%       1.00x
  Bull-SHOCK    2,703,415   3.184%  96.214%       2.21x
  Bear-LOW        26,346  43.806%  38.347%       1.00x
  Bear-SHOCK    2,924,295   3.086%  96.363%       2.28x

Shock bars (>= 99th): 5,627,710 (99.07%)
Sell in shock: 176,319 (3.13%)


## 5. Summary + Recommendations

In [7]:
print("="*65)
print("PHASE 4 EXPLORATION -- VERDICTS")
print("="*65)
print(f"{'Feature':<20} {'KS':>8} {'MI':>10} {'Redund':>8}  Verdict")
print("-"*65)
for name, r in results.items():
    ks = max(r['ks_sell'], r['ks_buy'])
    print(f"  {name:<18} {ks:>8.4f} {r['mi']:>10.6f} {r['ratio']:>8.2f}  {r['verdict']}")

print('''
PATCH RECOMMENDATIONS
=====================
1. gkv_norm:
   SUPERVISED -> replace vol_zscore in precompute_features.py, rebuild NPZ
   RL OBS     -> add as obs[15] (disabled VIO slot) in train_rl.py
   EXCLUDE    -> keep vol_zscore

2. d_atr_norm (ALWAYS apply to deploy gate regardless of MI):
   In paper_trade.py get_regime():
     OLD: vol_enc = 1.0 if atr_recent > atr_baseline*1.4 else 0.0
     NEW: vol_enc = 1.0 if (atr_recent > atr_baseline*1.4
                            and atr_recent > atr_baseline) else 0.0
   Releases gate during Bear+HIGH falling-ATR recovery periods.

3. momentum_5bar:
   SUPERVISED -> add as 11th feature, rebuild NPZ
   EXCLUDE    -> ret_15m in RL obs already covers this

4. DHPF 4-partition sampler (add to train_supervised.py):
   Bear-SHOCK (vol >= 99th pct) -> x4.0 multiplier
   Add to build_regime_balanced_sampler():
     shock_mask = vol_enc_raw >= np.percentile(vol_enc_raw, 99)
     regime_mult[bear_mask & shock_mask] = 4.0

5. Silver futures (XAGUSD):
   Requires separate MT5 feed + NPZ rebuild.
   Run 12 candidate after Run 11 RMSNorm.
''')


PHASE 4 EXPLORATION -- VERDICTS
Feature                    KS         MI   Redund  Verdict
-----------------------------------------------------------------
  gkv_norm             0.0807   0.000598   360.94  EXCLUDE (MI fails)
  d_atr_norm           0.1489   0.049595   137.16  RL OBS
  momentum_5bar        0.1830   0.012213  1123.99  RL OBS

PATCH RECOMMENDATIONS
1. gkv_norm:
   SUPERVISED -> replace vol_zscore in precompute_features.py, rebuild NPZ
   RL OBS     -> add as obs[15] (disabled VIO slot) in train_rl.py
   EXCLUDE    -> keep vol_zscore

2. d_atr_norm (ALWAYS apply to deploy gate regardless of MI):
   In paper_trade.py get_regime():
     OLD: vol_enc = 1.0 if atr_recent > atr_baseline*1.4 else 0.0
     NEW: vol_enc = 1.0 if (atr_recent > atr_baseline*1.4
                            and atr_recent > atr_baseline) else 0.0
   Releases gate during Bear+HIGH falling-ATR recovery periods.

3. momentum_5bar:
   SUPERVISED -> add as 11th feature, rebuild NPZ
   EXCLUDE    -> ret_15